# Create and organize objects in Unity Catalog

## Demo Scenario: University Data Platform

**Industry Context: Higher Education**

In this demo, we'll build a comprehensive data platform for a modern university. The university needs to organize and govern data across multiple domains:

- **Student Information System**: Student profiles, enrollment records, academic history
- **Course Management**: Course catalog, class schedules, instructor assignments  
- **Academic Performance**: Grades, assessments, analytics for student success
- **Research & Compliance**: Faculty research data, regulatory reporting requirements

As Azure Databricks Data Engineers, we'll demonstrate how Unity Catalog helps us:
- Apply consistent naming conventions across educational data assets
- Create catalogs and schemas that align with university organizational structure
- Build tables, views, and volumes for structured and unstructured educational data
- Implement governance controls while enabling self-service analytics for faculty and administrators

Let's begin by setting up our sample education data.

### Setup: Load Sample Education Data

Run the following cell to create sample tables for our university demo:
- **students**: Student enrollment and demographic information
- **courses**: Course catalog with departments and instructors
- **enrollments**: Student course registrations by semester
- **assessments**: Grades and evaluation results

In [ ]:
# Run the setup script to create sample data

from scripts.setup import setup
setup(spark)

### 🎯 DEMO: Explore Sample Data

In [ ]:
%sql
-- Set our working context to the education schema
USE CATALOG trainer_demo;
USE SCHEMA demo_03;

-- Preview student data
SELECT * FROM students LIMIT 5;

In [ ]:
%sql
-- Preview course catalog
SELECT * FROM courses LIMIT 5;

In [ ]:
%sql
-- Preview enrollment data
SELECT 
  e.enrollment_id,
  s.first_name,
  s.last_name,
  c.course_name,
  e.semester,
  e.year,
  e.enrollment_status
FROM enrollments e
INNER JOIN students s ON e.student_id = s.student_id
INNER JOIN courses c ON e.course_id = c.course_id
LIMIT 10;

## Apply naming conventions

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.apply-naming-conventions.png)

### 🎯 DEMO: University Naming Strategy

For our university platform, we'll apply a consistent naming strategy following medallion architecture:

```txt
university_data (Catalog)
├── bronze (Schema – raw ingestion)
│   ├── raw_student_info_system
│   ├── raw_course_registrations
│   └── raw_learning_management_system
│
├── silver (Schema – cleaned and validated)
│   ├── students
│   ├── courses
│   ├── enrollments
│   └── assessments
│
└── gold (Schema – analytics-ready)
    ├── student_performance_summary
    ├── course_enrollment_trends
    ├── department_analytics
    └── vw_student_success_metrics
```

**Key naming conventions:**
- Use **lowercase with underscores** for readability
- **Prefix views** with `vw_` to distinguish from tables
- **Layer-based schemas** (bronze/silver/gold) for data quality tiers
- **Descriptive names** that indicate business purpose

#### 🎯 DEMO: Show Catalog and Schema Organization

In [ ]:
%sql
-- List all catalogs in the metastore
SHOW CATALOGS;

In [ ]:
%sql
-- Show schemas within our training catalog
SHOW SCHEMAS IN trainer_demo;

In [ ]:
%sql
-- List all tables in the education schema
SHOW TABLES IN trainer_demo.demo_03;

## Create catalog

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/create-and-organize-objects-in-unity-catalog/media/3-catalog-definition.png)

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.create-catalog.png)

### 🎯 DEMO: Create University Catalogs

In this demo, we'll create catalogs to separate development and production environments for the university's data platform.

**Scenario**: The university needs separate environments to:
- Allow data engineers to experiment without affecting production systems
- Maintain strict access controls on student records in production
- Enable cost tracking and resource allocation by environment

In [ ]:
%sql
-- Create a development catalog for the university
CREATE CATALOG IF NOT EXISTS university_dev
COMMENT 'Development environment for university data engineering';

In [ ]:
%sql
-- Create a production catalog for the university
CREATE CATALOG IF NOT EXISTS university_prod
COMMENT 'Production environment for university analytics and reporting';

In [ ]:
%sql
-- View recently created catalogs
SELECT catalog_name
FROM system.information_schema.catalogs
WHERE catalog_name LIKE 'university%';

In [ ]:
%sql
-- Describe catalog to see properties
DESCRIBE CATALOG EXTENDED university_dev;

#### 🎯 DEMO: Using Catalog Explorer

**Trainer Note**: Switch to the UI to demonstrate Catalog Explorer:
1. Click **Catalog** in the left sidebar
2. Show the newly created `university_dev` and `university_prod` catalogs
3. Explain the hierarchical structure: Catalog → Schema → Table
4. Point out the automatically generated `default` and `information_schema` schemas
5. Discuss workspace bindings and how to restrict catalog access to specific workspaces

### 🎯 DEMO: Create University Catalogs

In this demo, we'll create catalogs to separate development and production environments for the university's data platform.

**Scenario**: The university needs separate environments to:
- Allow data engineers to experiment without affecting production systems
- Maintain strict access controls on student records in production
- Enable cost tracking and resource allocation by environment


## Create schema

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.create-schema.png)

### 🎯 DEMO: Create Schemas for Medallion Architecture

Create bronze, silver, and gold schemas to organize data by processing stage.

**Scenario**: The university implements a medallion architecture:
- **Bronze**: Raw data from source systems (SIS, LMS, etc.)
- **Silver**: Cleaned, validated, and deduplicated data
- **Gold**: Aggregated analytics and reporting datasets

In [ ]:
%sql
-- Create bronze schema for raw ingestion
CREATE SCHEMA IF NOT EXISTS university_dev.bronze
COMMENT 'Raw data ingestion layer from university source systems';

In [ ]:
%sql
-- Create silver schema for cleaned data
CREATE SCHEMA IF NOT EXISTS university_dev.silver
COMMENT 'Cleaned and validated university data';

In [ ]:
%sql
-- Create gold schema for analytics
CREATE SCHEMA IF NOT EXISTS university_dev.gold
COMMENT 'Aggregated analytics and reporting datasets';

In [ ]:
%sql
-- List schemas in the development catalog
SHOW SCHEMAS IN university_dev;

In [ ]:
%sql
-- Set our working context to the silver schema
USE CATALOG university_dev;
USE SCHEMA silver;

-- Confirm current context
SELECT current_catalog(), current_schema();

## Create tables and views

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.create-tables-views-materialized.png)

### 🎯 DEMO: Create Managed Tables

Create managed tables in the silver schema by copying data from our demo_03 schema.

**Scenario**: We'll move student and course data into the silver layer with proper schema definitions and constraints.

In [ ]:
%sql
-- Create managed table for students with primary key
CREATE TABLE IF NOT EXISTS university_dev.silver.students (
  student_id INT NOT NULL,
  first_name STRING,
  last_name STRING,
  email STRING,
  enrollment_date DATE,
  program STRING,
  year_level INT,
  gpa DOUBLE,
  CONSTRAINT students_pk PRIMARY KEY (student_id)
)
COMMENT 'Student enrollment and demographic information'
TBLPROPERTIES ('quality' = 'silver', 'domain' = 'student_records');

In [ ]:
%sql
-- Populate the table with data from our demo schema
INSERT INTO university_dev.silver.students
SELECT * FROM trainer_demo.demo_03.students;

In [ ]:
%sql
-- Create courses table
CREATE TABLE IF NOT EXISTS university_dev.silver.courses (
  course_id STRING NOT NULL,
  course_name STRING,
  department STRING,
  credits INT,
  level STRING,
  instructor STRING,
  CONSTRAINT courses_pk PRIMARY KEY (course_id)
)
COMMENT 'University course catalog';

In [ ]:
%sql
-- Populate courses table
INSERT INTO university_dev.silver.courses
SELECT * FROM trainer_demo.demo_03.courses;

In [ ]:
%sql
-- Create enrollments table with foreign keys
CREATE TABLE IF NOT EXISTS university_dev.silver.enrollments (
  enrollment_id INT NOT NULL,
  student_id INT,
  course_id STRING,
  semester STRING,
  year INT,
  enrollment_status STRING,
  CONSTRAINT enrollments_pk PRIMARY KEY (enrollment_id),
  CONSTRAINT fk_enrollment_student FOREIGN KEY (student_id) REFERENCES university_dev.silver.students(student_id),
  CONSTRAINT fk_enrollment_course FOREIGN KEY (course_id) REFERENCES university_dev.silver.courses(course_id)
)
COMMENT 'Student course enrollments by semester';

In [ ]:
%sql
-- Populate enrollments
INSERT INTO university_dev.silver.enrollments
SELECT * FROM trainer_demo.demo_03.enrollments;

In [ ]:
%sql
-- Verify table creation
SHOW TABLES IN university_dev.silver;

### 🎯 DEMO: Create Standard Views

Create views that simplify common queries and provide business-friendly interfaces.

**Scenario**: Faculty and administrators need easy access to enrollment statistics without writing complex SQL.

In [ ]:
%sql
-- Create view for active enrollments with student and course details
CREATE OR REPLACE VIEW university_dev.silver.vw_active_enrollments AS
SELECT 
  e.enrollment_id,
  e.student_id,
  s.first_name,
  s.last_name,
  s.email,
  s.program,
  c.course_id,
  c.course_name,
  c.department,
  c.instructor,
  e.semester,
  e.year,
  e.enrollment_status
FROM university_dev.silver.enrollments e
INNER JOIN university_dev.silver.students s ON e.student_id = s.student_id
INNER JOIN university_dev.silver.courses c ON e.course_id = c.course_id
WHERE e.enrollment_status = 'Enrolled';

In [ ]:
%sql
-- Query the view
SELECT * FROM university_dev.silver.vw_active_enrollments LIMIT 10;

In [ ]:
%sql
-- Create view for student performance summary
CREATE OR REPLACE VIEW university_dev.silver.vw_student_gpa_summary AS
SELECT 
  program,
  year_level,
  COUNT(*) as student_count,
  ROUND(AVG(gpa), 2) as avg_gpa,
  ROUND(MIN(gpa), 2) as min_gpa,
  ROUND(MAX(gpa), 2) as max_gpa
FROM university_dev.silver.students
GROUP BY program, year_level
ORDER BY program, year_level;

In [ ]:
%sql
-- Query the GPA summary view
SELECT * FROM university_dev.silver.vw_student_gpa_summary;

### 🎯 DEMO: Create Gold Layer Aggregations

Create gold-layer tables with pre-aggregated business metrics.

**Scenario**: The university's leadership dashboard requires daily-refreshed enrollment metrics.

In [ ]:
%sql
-- Create gold table for department enrollment summary
CREATE OR REPLACE TABLE university_dev.gold.department_enrollment_summary AS
SELECT 
  c.department,
  e.semester,
  e.year,
  COUNT(DISTINCT e.student_id) as unique_students,
  COUNT(e.enrollment_id) as total_enrollments,
  COUNT(DISTINCT e.course_id) as courses_offered
FROM university_dev.silver.enrollments e
INNER JOIN university_dev.silver.courses c ON e.course_id = c.course_id
WHERE e.enrollment_status IN ('Enrolled', 'Completed')
GROUP BY c.department, e.semester, e.year;

In [ ]:
%sql
-- Query the gold aggregation
SELECT * FROM university_dev.gold.department_enrollment_summary
ORDER BY year DESC, semester, department;

### 🎯 DEMO: External Tables

**Trainer Note**: External tables are useful when you need to query data stored outside Unity Catalog's managed storage. In a university setting, this could be:
- Historical data still stored in data lake containers
- Shared datasets from research collaborations
- Archived records that must remain in specific storage locations for compliance

**Example scenario**: The university has archived graduation records in ADLS Gen2 that need to remain in their original location but should be queryable through Unity Catalog.

```sql
-- Example: Create external table (requires pre-configured external location)
CREATE EXTERNAL TABLE IF NOT EXISTS university_dev.bronze.archived_graduates
LOCATION 'abfss://archives@universitydata.dfs.core.windows.net/graduates/'
COMMENT 'Historical graduation records in external storage';
```

*Note to trainer: This requires an external location to be configured in Unity Catalog first.*

## Create volumes

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.create-volumes.png)

### 🎯 DEMO: Create Managed Volumes

Create managed volumes for non-tabular files like syllabi, course materials, and student documents.

**Scenario**: The university needs governed storage for:
- Course syllabi (PDF files)
- Assignment submissions (various formats)
- Machine learning models for student success prediction
- Images and multimedia content for courses

In [ ]:
%sql
-- Create volume for course materials
CREATE VOLUME IF NOT EXISTS university_dev.silver.course_materials
COMMENT 'Storage for course syllabi, assignments, and learning resources';

In [ ]:
%sql
-- Create volume for ML models
CREATE VOLUME IF NOT EXISTS university_dev.gold.ml_models
COMMENT 'Trained machine learning models for student analytics';

In [ ]:
%sql
-- List volumes in the silver schema
SHOW VOLUMES IN university_dev.silver;

#### 🎯 DEMO: Working with Files in Volumes

Use Python to demonstrate file operations within volumes.

In [ ]:
# Get the volume path
volume_path = "/Volumes/university_dev/silver/course_materials"

# List files in the volume (will be empty initially)
files = dbutils.fs.ls(volume_path)
print(f"Files in course_materials volume: {len(files)}")
for file in files:
    print(f"  - {file.name}")

In [ ]:
# Create a sample syllabus file
sample_syllabus = """
COURSE SYLLABUS
Course: CS101 - Introduction to Programming
Instructor: Dr. Sarah Mitchell
Semester: Fall 2024

Course Description:
This course introduces fundamental programming concepts using Python.

Grading:
- Assignments: 40%
- Midterm: 25%
- Final Project: 35%
"""

# Write the file to the volume
syllabus_path = f"{volume_path}/CS101_syllabus.txt"
dbutils.fs.put(syllabus_path, sample_syllabus, overwrite=True)

print(f"Created sample syllabus at: {syllabus_path}")

In [ ]:
# Read the file back
content = dbutils.fs.head(syllabus_path)
print("Syllabus content:")
print(content)

In [ ]:
# Demonstrate using volumes with pandas
import pandas as pd

# Create a sample dataframe with student feedback
feedback_data = {
    'course_id': ['CS101', 'CS201', 'DE101', 'BA101'],
    'semester': ['Fall 2024', 'Fall 2024', 'Spring 2024', 'Fall 2024'],
    'avg_rating': [4.5, 4.2, 4.8, 4.3],
    'response_count': [45, 38, 52, 41]
}

df = pd.DataFrame(feedback_data)

# Save as CSV in the volume
csv_path = f"{volume_path}/course_feedback.csv"
df.to_csv(csv_path, index=False)

print("Saved course feedback CSV to volume")
print(df)

In [ ]:
%sql
-- Query CSV directly using SQL
SELECT * FROM csv.`/Volumes/university_dev/silver/course_materials/course_feedback.csv`;

### 🎯 DEMO: External Volumes

**Trainer Note**: External volumes provide governance for existing cloud storage locations without moving the data.

**Example scenario**: The university's Learning Management System (LMS) exports student submissions to an Azure Blob Storage container. We want to add Unity Catalog governance without moving the files.

```sql
-- Example: Create external volume (requires pre-configured external location)
CREATE EXTERNAL VOLUME IF NOT EXISTS university_dev.bronze.lms_exports
LOCATION 'abfss://lms-data@universitydata.dfs.core.windows.net/exports/'
COMMENT 'LMS system exports - externally managed storage';
```

*Note to trainer: This requires an external location with appropriate credentials configured first.*

## Implement DDL operations

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.implement-ddl-operations.png)

### 🎯 DEMO: Create Reusable Functions

Create SQL functions that encapsulate university-specific business logic.

**Scenario**: The university needs standardized calculations for:
- GPA letter grade conversion
- Credit requirements by degree program
- Academic standing determination

In [ ]:
%sql
-- Create function to convert numeric GPA to letter grade
CREATE OR REPLACE FUNCTION university_dev.silver.gpa_to_letter_grade(gpa DOUBLE)
RETURNS STRING
COMMENT 'Convert numeric GPA (0-4.0 scale) to letter grade'
RETURN CASE
  WHEN gpa >= 3.7 THEN 'A'
  WHEN gpa >= 3.3 THEN 'A-'
  WHEN gpa >= 3.0 THEN 'B+'
  WHEN gpa >= 2.7 THEN 'B'
  WHEN gpa >= 2.3 THEN 'B-'
  WHEN gpa >= 2.0 THEN 'C+'
  WHEN gpa >= 1.7 THEN 'C'
  WHEN gpa >= 1.0 THEN 'D'
  ELSE 'F'
END;

In [ ]:
%sql
-- Test the function
SELECT 
  student_id,
  first_name,
  last_name,
  gpa,
  university_dev.silver.gpa_to_letter_grade(gpa) as letter_grade
FROM university_dev.silver.students
LIMIT 10;

In [ ]:
%sql
-- Create function to determine academic standing
CREATE OR REPLACE FUNCTION university_dev.silver.academic_standing(gpa DOUBLE)
RETURNS STRING
COMMENT 'Determine academic standing based on GPA'
RETURN CASE
  WHEN gpa >= 3.5 THEN 'Dean\'s List'
  WHEN gpa >= 3.0 THEN 'Good Standing'
  WHEN gpa >= 2.0 THEN 'Satisfactory'
  WHEN gpa >= 1.5 THEN 'Academic Warning'
  ELSE 'Academic Probation'
END;

In [ ]:
%sql
-- Use the function in a query
SELECT 
  academic_standing(gpa) as standing,
  COUNT(*) as student_count,
  ROUND(AVG(gpa), 2) as avg_gpa
FROM university_dev.silver.students
GROUP BY academic_standing(gpa)
ORDER BY avg_gpa DESC;

#### 🎯 DEMO: Python Functions

Create Python functions for more complex logic like text normalization.

In [ ]:
%sql
-- Create Python function to standardize email domains
CREATE OR REPLACE FUNCTION university_dev.silver.extract_email_domain(email STRING)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Extract domain from email address'
AS $$
  if email and '@' in email:
    return email.split('@')[1].lower()
  return None
$$;

In [ ]:
%sql
-- Use the Python function
SELECT 
  extract_email_domain(email) as email_domain,
  COUNT(*) as student_count
FROM university_dev.silver.students
GROUP BY extract_email_domain(email);

### 🎯 DEMO: ALTER TABLE Operations

Modify table structure to add new columns and update constraints.

**Scenario**: The university wants to track international student status and add graduation date tracking.

In [ ]:
%sql
-- Add a new column to the students table
ALTER TABLE university_dev.silver.students
ADD COLUMN is_international BOOLEAN
COMMENT 'Indicates if student is international';

In [ ]:
%sql
-- Add another column for expected graduation
ALTER TABLE university_dev.silver.students
ADD COLUMN expected_graduation_date DATE
COMMENT 'Expected graduation date based on program requirements';

In [ ]:
%sql
-- Update the new columns with sample data
UPDATE university_dev.silver.students
SET is_international = (student_id % 5 = 0),
    expected_graduation_date = DATE_ADD(enrollment_date, (4 - year_level) * 365 + 365);

In [ ]:
%sql
-- Verify the changes
SELECT student_id, first_name, last_name, year_level, is_international, expected_graduation_date
FROM university_dev.silver.students
LIMIT 10;

In [ ]:
%sql
-- Describe the table to see the updated schema
DESCRIBE TABLE EXTENDED university_dev.silver.students;

### 🎯 DEMO: ALTER CATALOG Operations

Modify catalog properties and ownership.

**Scenario**: Tag catalogs for cost tracking and enable predictive optimization for production.

In [ ]:
%sql
-- Add tags to catalog for governance
ALTER CATALOG university_dev SET TAGS (
  'environment' = 'development',
  'cost_center' = 'academic_affairs',
  'data_classification' = 'internal'
);  

In [ ]:
%sql
-- View catalog details to see applied tags
DESCRIBE CATALOG EXTENDED university_dev;

**Trainer Note**: Demonstrate additional ALTER operations:
- `ALTER SCHEMA` to modify schema properties
- `ALTER TABLE ... RENAME COLUMN` to rename columns (requires Delta Lake column mapping)
- `ALTER TABLE ... DROP COLUMN` to remove columns
- Table properties like `delta.enableChangeDataFeed` for CDC

## Implement foreign catalog

![diagram](https://learn.microsoft.com/en-us/training/wwl-databricks/create-and-organize-objects-in-unity-catalog/media/8-foreign-catalogs-reason.png)

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.implement-foreign-catalog.png)

### 🎯 DEMO: Foreign Catalog Concepts

**Trainer Note**: Foreign catalogs enable querying external databases directly through Unity Catalog without data migration.

**University Scenario**: 
The university's legacy Student Information System (SIS) runs on PostgreSQL. Rather than migrating terabytes of historical data, we can create a foreign catalog to query it directly while maintaining Unity Catalog governance.

**Benefits for the university**:
1. **Unified Governance**: All queries to PostgreSQL go through Unity Catalog's access controls
2. **Audit Trail**: All access to legacy data is logged in Unity Catalog
3. **No Data Duplication**: Legacy data stays in PostgreSQL, avoiding sync complexity
4. **Incremental Migration**: Access legacy data while gradually migrating to Databricks

**Implementation Steps** (for demonstration purposes):

```sql
-- Step 1: Create connection to external PostgreSQL database
CREATE CONNECTION university_sis_db TYPE POSTGRESQL
OPTIONS (
  host 'sis-database.university.edu',
  port '5432',
  user secret ('university-secrets', 'postgres-user'),
  password secret ('university-secrets', 'postgres-password')
);

-- Step 2: Create foreign catalog using the connection
CREATE FOREIGN CATALOG IF NOT EXISTS legacy_sis
USING CONNECTION university_sis_db
OPTIONS (database 'student_information_system')
COMMENT 'Legacy Student Information System on PostgreSQL';

-- Step 3: Query external data through Unity Catalog
SELECT * FROM legacy_sis.public.historical_transcripts
WHERE graduation_year >= 2010;

-- Step 4: Join foreign catalog data with Unity Catalog tables
SELECT 
  s.student_id,
  s.first_name,
  s.last_name,
  h.degree_program as previous_degree,
  h.graduation_year as previous_grad_year
FROM university_dev.silver.students s
INNER JOIN legacy_sis.public.alumni_history h 
  ON s.email = h.email
WHERE h.graduation_year >= 2015;
```

**Supported External Systems**:
- PostgreSQL
- MySQL
- SQL Server / Azure SQL Database
- Snowflake
- BigQuery
- Redshift

*Note: This demo requires external database connectivity and appropriate network configuration.*

## Configure AI/BI Genie instructions

![infographic](https://raw.githubusercontent.com/MicrosoftLearning/DP-750T00-Implement-Data-Engineering-Solutions-using-Azure-Databricks/refs/heads/main/Demo/infographics/create-and-organize-objects-in-unity-catalog.configure-ai-bi-genie-instructions.png)

### 🎯 DEMO: AI/BI Genie Configuration

**Trainer Note**: AI/BI Genie enables business users to query data using natural language. Proper configuration helps Genie understand university-specific terminology.

**University Scenario**:
Academic advisors and department chairs need to analyze enrollment trends but aren't familiar with SQL. AI/BI Genie lets them ask questions like "Show me enrollment trends by department" and get accurate results.

#### Configuration Steps (UI Demonstration)

**1. Create a Genie Space**
- Navigate to AI/BI in the Databricks workspace
- Create a new Genie space called "University Enrollment Analytics"
- Add tables: `students`, `courses`, `enrollments`, `vw_active_enrollments` (from the university_dev.silver schema)

**2. Configure the Knowledge Store**

**Table Metadata Enhancements**:
```
students table description:
"Contains student enrollment records including demographics, 
academic program, GPA, and enrollment dates. Use this table 
when analyzing student population or academic performance."

Key column synonyms:
- program: "major", "degree program", "field of study"
- year_level: "class year", "grade level", "academic year"
- gpa: "grade point average", "academic performance score"
```

**3. Define Relationships**:
```
enrollments.student_id = students.student_id (Many to One)
enrollments.course_id = courses.course_id (Many to One)
```

**4. Add SQL Instructions**:

```sql
-- Example Query 1: "Show enrollment trends by department"
SELECT 
  c.department,
  e.semester,
  e.year,
  COUNT(DISTINCT e.student_id) as student_count
FROM university_dev.silver.enrollments e
JOIN university_dev.silver.courses c ON e.course_id = c.course_id
WHERE e.enrollment_status = 'Enrolled'
GROUP BY c.department, e.semester, e.year
ORDER BY e.year DESC, e.semester, c.department;

-- Example Query 2: "Which students are on the Dean's List?"
SELECT 
  student_id,
  first_name,
  last_name,
  program,
  gpa
FROM university_dev.silver.students
WHERE gpa >= 3.5
ORDER BY gpa DESC;

-- Example Query 3: "Show courses with the highest enrollment"
SELECT 
  c.course_name,
  c.department,
  c.instructor,
  COUNT(e.enrollment_id) as enrollment_count
FROM university_dev.silver.courses c
JOIN university_dev.silver.enrollments e ON c.course_id = e.course_id
WHERE e.enrollment_status = 'Enrolled'
GROUP BY c.course_name, c.department, c.instructor
ORDER BY enrollment_count DESC
LIMIT 10;
```

**5. Define SQL Expressions**:
```
Name: Dean's List Students
Expression: gpa >= 3.5
Synonyms: "high achievers", "top students", "honors students"

Name: Active Enrollment
Expression: enrollment_status IN ('Enrolled', 'In Progress')
Synonyms: "current enrollment", "ongoing courses"
```

**6. Test Natural Language Queries**:
- "How many students are in each program?"
- "Show me Computer Science courses taught by Dr. Sarah Mitchell"
- "Which departments have the most enrollments this fall?"
- "Who are the Dean's List students in Data Engineering?"

#### Demo Benefits

**For Faculty and Advisors**:
- No SQL knowledge required
- Self-service analytics for enrollment and academic performance
- Quick answers to ad-hoc questions

**For Data Engineers**:
- Centralized knowledge management
- Reduced ad-hoc query requests
- Consistent business logic across queries

**Trainer Note**: Show the iterative refinement process:
1. Ask a question that fails
2. Review the generated SQL
3. Correct it and add as an instruction
4. Test the question again to show improvement

---

## Demo Summary

In this demo, we covered the complete lifecycle of creating and organizing objects in Unity Catalog through the lens of a university data platform:

✅ **Applied naming conventions** following medallion architecture  
✅ **Created catalogs** for environment isolation (dev/prod)  
✅ **Created schemas** to organize data by processing layer (bronze/silver/gold)  
✅ **Built tables and views** with proper constraints and relationships  
✅ **Created volumes** for non-tabular university files  
✅ **Implemented DDL operations** including functions, ALTER statements  
✅ **Explored foreign catalogs** for legacy system integration  
✅ **Configured AI/BI Genie** for natural language analytics  

**Key Takeaways for Azure Databricks Data Engineers**:

1. **Start with a clear naming strategy** aligned with your organization's structure
2. **Use catalogs for isolation** (environments, sensitivity, compliance)
3. **Leverage schemas to organize** data by domain or processing stage
4. **Define constraints** (PK/FK) even though not enforced - they help query optimization
5. **Create reusable functions** to standardize business logic
6. **Use volumes** for governing unstructured data alongside structured tables
7. **Configure Genie spaces** to enable self-service analytics for business users

The principles demonstrated here apply across industries - adapt the naming and organization to fit your specific business domain and governance requirements.